================================================================================
模块 10：量化回测避坑指南与实盘系统工程 · 第 3/4 课
第 3 课：现代量化数据管道：复权算法与 Point-in-Time 时序对齐校验 (Data Pipeline & PIT Alignment)
================================================================================

【课程定位与投决会核心心智模型】
::: callout-note
### 📌 投决会核心心智模型：垃圾进垃圾出，数据管道纯净度是量化系统的生命中枢
“Garbage In, Garbage Out”。量化策略的生与死，70% 决定于底层数据管道（Data Pipeline）的严密性与工程纯净度。本课聚焦量化数据工程的两大核心支柱：价格除权除息（前复权/后复权）的数学连续性，与 Point-in-Time (PIT) 财报时序对齐的因果物理隔离。任何在报告期截止日（3-31）直接取用财务指标的行为，都是对实盘未来信息的神谕盗窃；任何未做复权处理的日线回测，都会把机械性分红除权误判为断崖暴跌。
:::

- **[除权参考价与累计前复权因子法]**：现金分红与送转股引发的股价跳空属于纯会计除权，必须通过单期折算因子 $f_t = P_{\text{ex}, t} / P_{\text{close}, t-1}$ 向下折算历史价格，保持最新一天的绝对价格与交易所盘口完全保真。
- **[Point-in-Time 双时间戳不可变快照]**：工业级基本面数据湖必须分离‘业务归属期（Report Period）’与‘法定披露时间戳（Publish Time）’，只允许基于 $t_{\text{publish}} \le T_{\text{decision}}$ 进行向后非等值对齐（`merge_asof`）。
- **[异常数据防御网]**：数据流接入必须经过严格的断号重排、未来时间戳检测、时间戳单调性校验以及极端负值价格拦截。

【微观机理与三位一体就近图文剖析】

锚点指引：下图展示了除权除息机械跳空与累计前复权因子的折算传导机制。

```mermaid
flowchart TD
    Raw["原始未复权价格 (Raw Price)<br/>分红送股导致机械断崖跳空"] --> CalcEx["计算除权参考价: P_ex = (P_prev - D) / (1 + R_split)"]
    CalcEx --> Factor["单期折算因子: f_t = P_ex / P_prev"]
    Factor --> Accum["累计前复权因子: AdjFactor_t = Π_{k=t+1}^T f_k (最新日为 1.0)"]
    Accum --> Smooth["生成连续前复权价格: P_adj = P_raw × AdjFactor (均线指标消除假死叉)"]
```

- **[要点 1：最新价保真性]**：前复权保持最新交易日价格与实盘完全一致，使得当前生成的均线突破与止损触发价位可直接对齐交易所盘口。
- **[要点 2：指标假死叉的消除]**：若不进行前复权，10 送 10 除权会导致 5 日均线向下假破位，诱发策略错误清仓。
- **[要点 3：复权历史重写特性]**：前复权在每次新分红发生时会重写全部历史价格，因此长期 NAV 评价必须切换为后复权。

锚点指引：下图展示了 Point-in-Time (PIT) 数据湖双时序不可变快照架构。

```mermaid
flowchart LR
    A["财报披露源头: 报告期 2023-12-31 / 披露日 2024-04-20"] --> B["双时间戳存储: Valid Time (业务所属) + System Time (物理入湖)"]
    B --> C["时序快照追加: 若发生财报更正，以新版本不可变追加"]
    C --> D["AS-OF 历史切片查询: SELECT * WHERE publish_date <= T_decision"]
```

- **[要点 1：版本不可变性（Immutability）]**：对于发生差错更正的财报，严禁覆盖历史旧记录，必须以版本号（Version ID）追加存储。
- **[要点 2：非等值向后对齐（merge_asof）]**：在任意交易日 $T$，策略仅可见在此之前已依法公开披露的最新财务指标。
- **[要点 3：消灭幽灵超额（Ghost Alpha）]**：严格 PIT 对齐杜绝了在次年 1~4 月提前偷看年报净利润的神谕作弊。

【数学与估值模型】
除权参考价与单期折算因子公式：
$$
P_{\text{ex}, t} = \frac{P_{\text{close}, t-1} - D_t + P_{\text{rights}} \cdot R_{\text{rights}}}{1 + R_{\text{split}} + R_{\text{rights}}}, \quad f_t = \frac{P_{\text{ex}, t}}{P_{\text{close}, t-1}}
$$
累计前复权乘数方程（最新日 $T$ 为基准）：
$$
\text{AdjFactor}_T = 1.0, \quad \text{AdjFactor}_{\tau} = \prod_{k=\tau+1}^{T} f_k \quad (\tau < T)
$$
$$
P_{\text{adj}, \tau} = P_{\text{raw}, \tau} \times \text{AdjFactor}_{\tau}
$$
变量定义域与参数约束：
- $D_t \ge 0$：每股派发现金红利（元）；$R_{\text{split}} \ge 0$：每股送转股比例。
- $P_{\text{close}, t-1} > 0$：除权日前一日收盘价。
- $f_t \in (0, 1]$：单期折算因子。若当日无分红送转，$f_t = 1.0$。

Point-in-Time 数据可见性约束：
$$
\text{FundamentalFeature}_T = \operatorname{arg\,max}_{t_{\text{publish}} \le T} \{\text{FilingData}(t_{\text{report}}, t_{\text{publish}}, v)\}
$$

::: callout-warning
### ⚠️ 投决会风控红线与认知陷阱防范
1. **前复权回测长期复利的负价格荒谬性**：超长历史（如 20 年）前复权会导致早期股价被压缩至几分钱甚至负数，百分比涨跌幅严重扭曲，长期 CAGR 评估必须使用后复权。
2. **爬虫报告期主键前视陷阱**：开源数据源直接将 12-31 作为交易日主键关联财报，导致 4 个月的前视穿透，测出的价值因子全部是作弊假象。
3. **乱序行情破坏撮合因果链**：交易所 UDP 组播行情网络丢包抖动会导致接收时间与撮合时间倒挂，必须建立 Watermark 水位线重排机制。
:::

【思考题】
Q1: 为什么在做量化多因子回测与技术指标计算时必须使用“前复权（Backward Adjustment）”，而在统计长期投资组合资产净值（NAV）与复合年化收益率（CAGR）时更推荐使用“后复权（Forward Adjustment）”？
Q2: 很多开源量化库从通用财经网站爬取财务报表时，直接以“报告期（如 06-30、12-31）”作为时序主键。请说明这种做法会给中低频基本面量化策略带来几个月的“幽灵超额（Ghost Alpha）”，并在系统架构层面给出设计工业级 PIT 数据湖的元数据表结构方案。
Q3: 在高频 L2 Tick 行情数据管道中，深交所逐笔成交（Trade）与逐笔委托（Order）均带有严格单调递增的 `ApplSeqNum`（业务消息序列号），而上交所历史数据曾存在乱序广播与合并分包。面对跨交易所的网络抖动与乱序数据流，如何设计滑动窗口（Sliding Window）与水位线（Watermark）重排算法保证因果确定性？

【考核与通过规则】
- 代码填空并通过所有自动化单元测试断言：4 分
- Q1~Q3 问答题深入推导并符合投决会专业标准：各 2 分（共 6 分）
- 综合通过线：8/10 分


In [ ]:
from typing import List, Dict, Any
import numpy as np
import pandas as pd


def compute_backward_adjusted_prices(
    raw_market_df: pd.DataFrame,
    dividend_df: pd.DataFrame
) -> pd.DataFrame:
    """
    计算日线行情的精准累计前复权价格 (保持最新一天价格不变，历史价格向下折算):
    """
    # TODO: 实现完整前复权算法
    pass


def align_point_in_time_data(
    market_df: pd.DataFrame,
    fundamental_pit_df: pd.DataFrame
) -> pd.DataFrame:
    """
    将日线市场行情与 Point-in-Time 基本面财务数据严格对齐，杜绝因财报披露滞后导致的前视偏差。
    """
    # TODO: 实现 PIT 严格向后对齐
    pass


def validate_pipeline_integrity(df: pd.DataFrame) -> List[str]:
    """
    对清洗对齐后的量化数据集执行全量质检与未来信息反向扫描。
    """
    # TODO: 实现管道质检与异常模式扫描
    pass


if __name__ == "__main__":
    print("Lesson 03 (Data Pipeline & PIT) Stub Initialized.")
